In [6]:
import hdbscan
import umap
import bacpipe
import matplotlib.pyplot as plt
import numpy as np
import os
from pathlib import Path

In [7]:
import importlib.resources as pkg_resources
os.chdir(pkg_resources.files("bacpipe"))
os.chdir('..')

MODEL_NAME = 'aves_especies'                  # name of the model to run. Supported models are in bacpipe.supported_models
MARSH_DIR = Path("data/audio/")
SWAMP_DIR = Path("data/swamp")
# Must match the results directory used when training the probe.
RESULTS_DIR = bacpipe.settings.main_results_dir

In [8]:
bacpipe.settings.device = 'cuda' 
bacpipe.config.models = ['aves_especies']

In [12]:
os.getcwd()
os.chdir("/workspaces/OsaAnurans3.11")

In [13]:
# Generate or load embeddings for annotated marsh recordings
marsh_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(MARSH_DIR),
    check_if_already_processed=True,
)
marsh_embeds = marsh_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######



Finding all generated embeddings: 104it [00:00, 839.97it/s]
Found 104 embedding files.
finding audio files: 113it [00:00, 1513.07it/s]
Found 104 number of audio files.

### Embeddings already exist. Using embeddings in bacpipe_results/audio/embeddings/2026-10-15_20-41___aves_especies-audio ###


In [14]:
swamp_loader = bacpipe.run_pipeline_for_single_model(
    model_name=MODEL_NAME,
    audio_dir=str(SWAMP_DIR),
    check_if_already_processed=True,
)
swamp_embeds = swamp_loader.embeddings(return_type="array")

Checking if the selected models require a checkpoint, and if so, if the checkpoint already exists.

aves_especies checkpoint exists.




###### Generating embeddings using AVES_ESPECIES ######

Finding all generated embeddings: 277it [00:00, 18419.99it/s]
Found 277 embedding files.
finding audio files: 280it [00:00, 24434.70it/s]
Found 277 number of audio files.
The audio files are no longer where they used to be during the previous run. This might cause a problem.

### Embeddings already exist. Using embeddings in bacpipe_results/swamp/embeddings/2026-10-14_17-26___aves_especies-swamp ###


In [16]:
embeddings = np.concatenate([swamp_embeds, marsh_embeds], axis=0)
 

In [17]:
# 1. Cluster in the original embedding space
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=10,
    min_samples=5,
    metric="euclidean",
)

In [18]:
labels = clusterer.fit_predict(embeddings)

print(f"Number of clusters: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Noise points: {np.sum(labels == -1)}")



Number of clusters: 48
Noise points: 3105


/workspaces/OsaAnurans3.11/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
# 2. Reduce dimensions to 2D for visualization
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)

embedding_2d = reducer.fit_transform(embeddings)

In [22]:
# 3. Plot, using HDBSCAN's cluster labels
plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    embedding_2d[:, 0],
    embedding_2d[:, 1],
    c=labels,
    cmap="tab20",
    s=8,
    alpha=0.7,
)

plt.title("HDBSCAN clusters of audio embeddings")
plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")
plt.colorbar(scatter, label="Cluster ID (-1 = noise)")
plt.tight_layout()
plt.show()

In [27]:
import pandas as pd
import plotly.express as px

df = pd.DataFrame({
    "UMAP 1": embedding_2d[:, 0],
    "UMAP 2": embedding_2d[:, 1],
    "Cluster": labels.astype(str),
})

fig = px.scatter(
    df,
    x="UMAP 1",
    y="UMAP 2",
    color="Cluster",
    hover_data=["Cluster"],
    title="Interactive HDBSCAN clusters",
    render_mode="webgl",
)

fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.update_layout(
    width=1000,
    height=750,
    dragmode="pan",
)

fig.show(renderer="browser")
fig.write_html("hdbscan_clusters.html")